In [1]:
import torch
import torch.nn as nn

In [2]:

inputs = torch.tensor([
    [0.43,0.15,0.89], #Your->x^1
    [0.55,0.87,0.66], #jounrey->x^2
    [0.57,0.85,0.64], #starts->x^3
    [0.22,0.58,0.33], #with->x^4
    [0.77,0.25,0.10], #one->x^5
    [0.05,0.80,0.55] #step-> x^6
])

In [3]:
num_tokens = inputs.shape[0]
print(f"Number of Tokens: {num_tokens}")

Number of Tokens: 6


In [4]:
#Batching helps us provide the model with multiple inputs at once, hence reducing training time

batch = torch.stack((inputs,inputs),dim=0) 
print(batch.shape)

torch.Size([2, 6, 3])


In [8]:
class CausalAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,dropout, qkv_bias= False):

        super().__init__()
        self.d_out= d_out
        self.W_query = nn.Linear(d_in,d_out,bias= qkv_bias)
        self.W_key = nn.Linear(d_in,d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        #.register_buffer ensures that the tensors are in the same device, thus eliminating the chances of device mismatch error
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length), diagonal= 1)) 

    def forward(self,x):

        b,num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @keys.transpose(1,2) #Keeping the batch dimension same
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

        

In [9]:
#Using the Causal Self-attention Class

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in= inputs.shape[1], d_out =2,context_length=context_length,dropout= 0.0,qkv_bias= False)
context_vecs = ca(batch)
print(f"Context Vecs Shape: {context_vecs.shape}")

Context Vecs Shape: torch.Size([2, 6, 2])


## Multi-Head Self-Attention

In [10]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out,context_length, dropout, num_heads, qkv_bias = False):

        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in,d_out,context_length,dropout,qkv_bias) for _ in range(num_heads)]
        )

    def forward(self,x):

        return torch.cat([head(x) for head in self.heads], dim=-1) #Each head does a forward pass on the batch input separately

In [11]:
#Using th Multi Head  Attention Wrappper Class

torch.manual_seed(123)
context_length = batch.shape[1] #Number of tokens
d_in,d_out = 3,2
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs = mha(batch)


### More Efficient Multi-head Attention Class

In [12]:
#In the above implementation of multi-head attention class, we processed the attention modules sequentially using the for loop
#This is slow and computationally expensive
#instead we can compute the outputs for all attention heads simultaneously using Matrix Multiplication

In [22]:
class MultiHeadAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,dropout, num_heads, qkv_bias = False):

        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out//num_heads
        self.W_query = nn.Linear(d_in,d_out, bias = qkv_bias )
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    
    def forward(self,x):

        b, num_tokens,d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        #For keys, values and queries we implicitly split the matrix by adding a self.head_dim and unrolling b,num_tokens,d_out -> (b,num_tokens,num_heads,head_dim)
        keys = keys.view(b,num_tokens, self.num_heads,self.head_dim) #.view() reshapes a tensor without copying its underlying data in memory
        values = values.view(b,num_tokens,self.num_heads,self.head_dim) 
        queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)

        keys = keys.transpose(1,2) #transposing (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim) 
        queries = queries.transpose(1,2) #transposing (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim) 
        values = values.transpose(1,2) #transposing (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim) 


        attn_scores = queries @ keys.transpose(2,3) #dot product 
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens] #The masking matrix should be of the same dimension as the attention score matrix  

        attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
        attn_scores/keys.shape[-1] ** 0.5,dim=-1            
        )

        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1,2)

        context_vec = context_vec.contiguous().view(b,num_tokens, self.d_out) #COmbines the heads where self.d_out = self.num_heads * self.head_dim

        context_vec = self.out_proj(context_vec) #Adds an optional linear projection

        return context_vec

        

        
        
        

In [23]:
#Using the MultiHeadAttention CLass 

torch.manual_seed(123)
batch_size, context_length,d_in = batch.shape

d_out = 2

mha = MultiHeadAttention(d_in, d_out, context_length,0.0,num_heads=2)

context_vecs = mha(batch)

print(context_vecs)

print(f"context_vecs.shape: {context_vecs.shape}")

tensor([[[0.2595, 0.4014],
         [0.2583, 0.4014],
         [0.2583, 0.4014],
         [0.2575, 0.4031],
         [0.2582, 0.4026],
         [0.2575, 0.4028]],

        [[0.2595, 0.4014],
         [0.2583, 0.4014],
         [0.2583, 0.4014],
         [0.2575, 0.4031],
         [0.2582, 0.4026],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
